# Cosmos-T1 Top1-Top2 Cevap Doğruluğu Tahmini

Bu notebook, Cosmos-T1 modelinin GSM8K-TR sorularına verdiği cevapların doğru/yanlış olmasını, cevap üretimi sırasında oluşan token bazlı top1-top2 olasılık farklarından tahmin etmek için hazırlanmıştır.

Akış:
1. GSM8K-TR veri kümesini hazırla.
2. Cosmos-T1 ile cevap üret ve her token için top1-top2 farkını kaydet.
3. Fark dizilerinden istatistiksel özellikler çıkar.
4. Logistic Regression, Random Forest ve Gradient Boosting modellerini eğit/test et.
5. Sonuç tablolarını ve grafikleri kaydet.

In [ ]:
!pip install -q transformers accelerate datasets scikit-learn matplotlib seaborn

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/HA_but'
DATA_PATH = f'{ROOT}/cosmos_t1_kayitlar_768.jsonl'
RESULTS_DIR = f'{ROOT}/sonuclar'
FIGURES_DIR = f'{ROOT}/figures'

os.makedirs(ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print('ROOT:', ROOT)

In [ ]:
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList

random.seed(42)

MODEL_ID = 'ytu-ce-cosmos/Turkish-Gemma-9b-T1'
MAX_NEW_TOKENS = 1024

## 1. Veri Kümesini Hazırlama

`gsm8k_tr` cevapları düz Türkçe metin biçimindedir. Bu yüzden altın cevap son satırlardaki sayısal ifadeden çıkarılır. Saat biçimli cevaplar (`10:30` gibi) sayı çıkarımında hataya neden olabileceği için filtrelenir.

In [ ]:
def sayi_temizle(s):
    s = str(s).strip()
    s = re.sub(r"[^\d,.\-]", "", s)
    if s == "":
        return None

    if "." in s and "," in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        parts = s.split(",")
        if len(parts[-1]) == 3:
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
    elif "." in s:
        parts = s.split(".")
        if len(parts[-1]) == 3 and len(parts) > 1:
            s = s.replace(".", "")

    try:
        return float(s)
    except ValueError:
        return None


def gold_cevap(text):
    text = str(text)

    m = re.search(r"####\s*(-?\d+(?:[.,]\d+)*)", text)
    if m:
        return sayi_temizle(m.group(1))

    satirlar = [s.strip() for s in text.strip().split("\n") if s.strip()]
    for satir in reversed(satirlar):
        if re.search(r"\b\d{1,2}:\d{2}\b", satir):
            return None

        sayilar = re.findall(r"-?\d+(?:[.,]\d+)*", satir)
        if sayilar:
            return sayi_temizle(sayilar[-1])

    return None


def pred_cevap(text):
    text = str(text)

    m = re.search(r"####\s*(-?\d+(?:[.,]\d+)*)", text)
    if m:
        return sayi_temizle(m.group(1))

    m = re.search(
        r"(?:cevap|sonuç|sonuc|nihai cevap)[^\d\-]*(-?\d+(?:[.,]\d+)*)",
        text.lower(),
    )
    if m:
        return sayi_temizle(m.group(1))

    sayilar = re.findall(r"-?\d+(?:[.,]\d+)*", text)
    if sayilar:
        return sayi_temizle(sayilar[-1])

    return None


def veri_yukle():
    ds = load_dataset('ytu-ce-cosmos/gsm8k_tr')['train']
    sorular = []
    atlanan_saat = 0

    for i, ex in enumerate(ds):
        if re.search(r"\b\d{1,2}:\d{2}\b", str(ex['answer'])):
            atlanan_saat += 1
            continue

        gold = gold_cevap(ex['answer'])
        if gold is None:
            continue

        sorular.append({
            'id': i,
            'soru': ex['question'].strip(),
            'gold': gold,
            'gold_text': ex['answer'],
        })

    random.shuffle(sorular)
    print('Toplam veri:', len(ds))
    print('Temiz soru:', len(sorular))
    print('Saat formatlı atlanan:', atlanan_saat)
    return sorular

sorular = veri_yukle()

## 2. Modeli Yükleme

Bu hücre A100/L4 gibi GPU ortamında çalıştırılmalıdır. CPU üzerinde pratik değildir.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()

terminators = [tokenizer.eos_token_id]
end_turn = tokenizer.convert_tokens_to_ids('<end_of_turn>')
if isinstance(end_turn, int) and end_turn >= 0:
    terminators.append(end_turn)
terminators = list(set(terminators))

print('Model hazır:', MODEL_ID)
print('Terminators:', terminators)
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 3. Top1-Top2 Farkını Kaydetme

`output_scores=True` tüm vocabulary skorlarını sakladığı için bellek tüketimi yüksektir. Burada `LogitsProcessor` ile her adımda yalnızca top1-top2 farkı kaydedilir.

In [ ]:
class Top1Top2Kaydedici(LogitsProcessor):
    def __init__(self):
        self.farklar = []

    def __call__(self, input_ids, scores):
        probs = torch.softmax(scores.float(), dim=-1)
        top2 = torch.topk(probs, k=2, dim=-1).values
        fark = (top2[:, 0] - top2[:, 1]).detach().cpu().tolist()
        self.farklar.append(fark)
        return scores


def prompt_hazirla(soru):
    return (
        'Sadece kısa cevap ver. '
        'Adım adım uzun çözüm yazma. '
        'Son satır: #### <sayı>

'
        f'Soru: {soru}'
    )


@torch.no_grad()
def uret(soru):
    messages = [{'role': 'user', 'content': prompt_hazirla(soru)}]
    chat_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)
    kaydedici = Top1Top2Kaydedici()

    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=terminators,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        logits_processor=LogitsProcessorList([kaydedici]),
        output_scores=False,
        return_dict_in_generate=True,
    )

    prompt_len = inputs['input_ids'].shape[1]
    gen_ids = out.sequences[0][prompt_len:]
    metin = tokenizer.decode(gen_ids, skip_special_tokens=True)
    diffs = [float(x[0]) for x in kaydedici.farklar]

    del inputs, out, gen_ids
    torch.cuda.empty_cache()

    return metin, diffs

## 4. Checkpoint'li Veri Toplama

Bu hücre, daha önce kaydedilen kayıtları okur ve kaldığı yerden devam eder. Hedef değerler zaman/GPU durumuna göre değiştirilebilir.


**Not:** Veri toplama hücresi uzun sürdüğü için bilinçli olarak durdurulabilir. Kayıtlar `KAYIT_ARALIGI` değerine göre JSONL dosyasına ara ara yazılır. Hücre tekrar çalıştırıldığında `islenen_idler` üzerinden daha önce işlenen sorular atlanır ve süreç kaldığı yerden devam eder. Bu nedenle notebook çıktılarında görülen `KeyboardInterrupt` kayıt dosyası bozulduğu anlamına gelmez.


In [ ]:
def kayitlari_oku(path):
    if not os.path.exists(path):
        return []
    kayitlar = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                kayitlar.append(json.loads(line))
    return kayitlar


def kayitlari_yaz(path, yeni_kayitlar):
    if not yeni_kayitlar:
        return
    with open(path, 'a', encoding='utf-8') as f:
        for kayit in yeni_kayitlar:
            f.write(json.dumps(kayit, ensure_ascii=False) + '
')


HEDEF_DOGRU = 550
HEDEF_YANLIS = 550
N_RUN = 2000
KAYIT_ARALIGI = 10

kayitlar = kayitlari_oku(DATA_PATH)
islenen_idler = set(r['id'] for r in kayitlar)
dogru_sayisi = sum(r['dogru'] for r in kayitlar)
yanlis_sayisi = len(kayitlar) - dogru_sayisi
bekleyen_kayitlar = []

print('Önceki kayıt:', len(kayitlar))
print('Doğru:', dogru_sayisi, 'Yanlış:', yanlis_sayisi)

basla = time.time()

for q in sorular[:N_RUN]:
    if q['id'] in islenen_idler:
        continue

    metin, diffs = uret(q['soru'])
    pred = pred_cevap(metin)
    dogru = pred is not None and abs(pred - q['gold']) < 1e-4

    kayit = {
        'id': q['id'],
        'soru': q['soru'],
        'gold': q['gold'],
        'pred': pred,
        'dogru': dogru,
        'diffs': diffs,
        'n_token': len(diffs),
        'metin_kismi': metin[:500],
        'metin_son': metin[-500:],
    }

    bekleyen_kayitlar.append(kayit)
    islenen_idler.add(q['id'])
    dogru_sayisi += int(dogru)
    yanlis_sayisi += int(not dogru)

    print(
        f"[{len(islenen_idler)}] gold={q['gold']} pred={pred} "
        f"{'DOĞRU' if dogru else 'YANLIŞ'} token={len(diffs)} | "
        f"D={dogru_sayisi} Y={yanlis_sayisi}"
    )

    if len(bekleyen_kayitlar) >= KAYIT_ARALIGI:
        kayitlari_yaz(DATA_PATH, bekleyen_kayitlar)
        bekleyen_kayitlar = []
        print('Ara kayıt yazıldı.')

    if dogru_sayisi >= HEDEF_DOGRU and yanlis_sayisi >= HEDEF_YANLIS:
        break

kayitlari_yaz(DATA_PATH, bekleyen_kayitlar)

son_kayitlar = kayitlari_oku(DATA_PATH)
print('
Toplam kayıt:', len(son_kayitlar))
print('Doğru:', sum(r['dogru'] for r in son_kayitlar))
print('Yanlış:', len(son_kayitlar) - sum(r['dogru'] for r in son_kayitlar))
print(f'Süre: {(time.time() - basla)/60:.1f} dk')

## 5. Özellik Çıkarma

Her soru için değişken uzunluklu top1-top2 fark dizisi, 14 sabit özellikten oluşan bir vektöre dönüştürülür.

In [ ]:
def ozellik_cikar(diffs):
    x = np.array(diffs, dtype=float)
    if len(x) == 0:
        x = np.array([0.0])

    ilk10 = x[:10]
    son10 = x[-10:]

    if len(x) > 1:
        t = np.arange(len(x))
        trend = np.polyfit(t, x, 1)[0]
    else:
        trend = 0.0

    return {
        'n_tokens': len(x),
        'mean': np.mean(x),
        'std': np.std(x),
        'min': np.min(x),
        'max': np.max(x),
        'median': np.median(x),
        'q25': np.quantile(x, 0.25),
        'q75': np.quantile(x, 0.75),
        'iqr': np.quantile(x, 0.75) - np.quantile(x, 0.25),
        'mean_first10': np.mean(ilk10),
        'mean_last10': np.mean(son10),
        'prop_low_01': np.mean(x < 0.10),
        'prop_low_02': np.mean(x < 0.20),
        'trend': trend,
    }

kayitlar = kayitlari_oku(DATA_PATH)

satirlar = []
for r in kayitlar:
    o = ozellik_cikar(r['diffs'])
    o['dogru'] = int(r['dogru'])
    o['id'] = r['id']
    satirlar.append(o)

df = pd.DataFrame(satirlar)

print('Toplam kayıt:', len(df))
print('Doğru:', int(df['dogru'].sum()))
print('Yanlış:', int(len(df) - df['dogru'].sum()))
display(df.head())

df.to_csv(f'{RESULTS_DIR}/ozellikler_tum.csv', index=False)

## 6. Eğitim ve Test

Mevcut deneyde dengeli test kümesi 50 doğru + 50 yanlış olacak şekilde 100 örnekten oluşturulmuştur. Kalan dengeli örnekler eğitim için kullanılmıştır.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

N_TEST_CLASS = 50

dogru_sayi = int(df['dogru'].sum())
yanlis_sayi = int(len(df) - dogru_sayi)
N_CLASS = min(dogru_sayi, yanlis_sayi)
N_TRAIN_CLASS = N_CLASS - N_TEST_CLASS

if N_TRAIN_CLASS <= 0:
    raise ValueError('Dengeli train/test ayrımı için yeterli veri yok.')

dogru_df = df[df['dogru'] == 1].sample(n=N_CLASS, random_state=42)
yanlis_df = df[df['dogru'] == 0].sample(n=N_CLASS, random_state=42)

dogru_train = dogru_df.sample(n=N_TRAIN_CLASS, random_state=42)
yanlis_train = yanlis_df.sample(n=N_TRAIN_CLASS, random_state=42)

dogru_test = dogru_df.drop(dogru_train.index).sample(n=N_TEST_CLASS, random_state=42)
yanlis_test = yanlis_df.drop(yanlis_train.index).sample(n=N_TEST_CLASS, random_state=42)

df_train = pd.concat([dogru_train, yanlis_train]).sample(frac=1, random_state=42).reset_index(drop=True)
df_test = pd.concat([dogru_test, yanlis_test]).sample(frac=1, random_state=42).reset_index(drop=True)

ozellikler = [c for c in df.columns if c not in ['dogru', 'id']]
X_train = df_train[ozellikler]
y_train = df_train['dogru']
X_test = df_test[ozellikler]
y_test = df_test['dogru']

print('Train:', X_train.shape, 'Doğru:', int(y_train.sum()), 'Yanlış:', int(len(y_train) - y_train.sum()))
print('Test :', X_test.shape, 'Doğru:', int(y_test.sum()), 'Yanlış:', int(len(y_test) - y_test.sum()))

modeller = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

sonuclar = []
tahminler = {}

for ad, model_ml in modeller.items():
    model_ml.fit(X_train, y_train)
    pred = model_ml.predict(X_test)
    prob = model_ml.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, pred)

    sonuclar.append({
        'Model': ad,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, prob),
    })

    tahminler[ad] = {'pred': pred, 'prob': prob, 'cm': cm}
    print('
' + ad)
    print(cm)

sonuc_df = pd.DataFrame(sonuclar)
display(sonuc_df)

df_train.to_csv(f'{RESULTS_DIR}/train_240.csv', index=False)
df_test.to_csv(f'{RESULTS_DIR}/test_100.csv', index=False)
sonuc_df.to_csv(f'{RESULTS_DIR}/model_sonuclari_test100.csv', index=False)

## 7. Grafikler

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

plt.figure(figsize=(7, 4))
plt.hist(df_test[df_test['dogru'] == 1]['mean'], alpha=0.65, label='Doğru cevap', color='seagreen', bins=12)
plt.hist(df_test[df_test['dogru'] == 0]['mean'], alpha=0.65, label='Yanlış cevap', color='indianred', bins=12)
plt.xlabel('Ortalama Top1-Top2 Farkı')
plt.ylabel('Soru Sayısı')
plt.title('Doğru ve Yanlış Cevaplarda Ortalama Top1-Top2 Farkı')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_mean_diff_distribution.png', dpi=200)
plt.show()

plt.figure(figsize=(6, 5))
for ad in modeller:
    fpr, tpr, _ = roc_curve(y_test, tahminler[ad]['prob'])
    auc = roc_auc_score(y_test, tahminler[ad]['prob'])
    plt.plot(fpr, tpr, label=f'{ad} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Rastgele')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Eğrileri')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_roc.png', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ad in zip(axes, modeller):
    sns.heatmap(
        tahminler[ad]['cm'],
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        xticklabels=['Yanlış', 'Doğru'],
        yticklabels=['Yanlış', 'Doğru'],
        ax=ax,
    )
    ax.set_title(ad)
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_confusion_matrix.png', dpi=200)
plt.show()

rf_imp = pd.Series(modeller['Random Forest'].feature_importances_, index=ozellikler).sort_values()
gb_imp = pd.Series(modeller['Gradient Boosting'].feature_importances_, index=ozellikler).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_imp.plot(kind='barh', ax=axes[0], color='seagreen')
axes[0].set_title('Random Forest - Feature Importance')
gb_imp.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Gradient Boosting - Feature Importance')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_feature_importance.png', dpi=200)
plt.show()

# Token bazlı örnek grafik
test_ids = set(df_test['id'].tolist())
dogru_ornek = None
yanlis_ornek = None
for r in kayitlar:
    if r['id'] in test_ids and r['dogru'] is True and dogru_ornek is None:
        dogru_ornek = r
    if r['id'] in test_ids and r['dogru'] is False and yanlis_ornek is None:
        yanlis_ornek = r
    if dogru_ornek is not None and yanlis_ornek is not None:
        break

plt.figure(figsize=(10, 4))
if dogru_ornek is not None:
    plt.plot(dogru_ornek['diffs'], label=f"Doğru örnek (gold={dogru_ornek['gold']}, pred={dogru_ornek['pred']})", color='seagreen')
if yanlis_ornek is not None:
    plt.plot(yanlis_ornek['diffs'], label=f"Yanlış örnek (gold={yanlis_ornek['gold']}, pred={yanlis_ornek['pred']})", color='indianred', alpha=0.85)
plt.xlabel('Token Sırası')
plt.ylabel('Top1-Top2 Farkı')
plt.title('Token Bazlı Top1-Top2 Fark Dizisi')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_token_level_diff.png', dpi=200)
plt.show()

## Sonuç Özeti

Son deneyde 357 kayıt içinden dengeli olarak 240 eğitim ve 100 test örneği kullanılmıştır. Test kümesi 50 doğru + 50 yanlış olacak şekilde oluşturulmuştur. En dengeli sonuç Gradient Boosting modeli ile elde edilmiştir.